# Notebook 06 — Results and GeoAI: The Full Picture
**GeoAI for Climate, Environment and Health**

---

> **Where you are in the pipeline**
>
> ```
> 📋 Health records → 🛰 Satellite → 🌍 Environment → 🔗 Link → ⏱ Timing → [🎯 Results & GeoAI]
> ```
>
> This is where the pipeline pays off. You have birth records linked to satellite climate,
> with trimester-specific exposures and a solid understanding of the data's provenance.
> Now you will map spatial patterns, test associations statistically, and use a
> **natural-language GeoAI assistant** to interrogate the dataset.

---

In [ ]:
%%html
<div id="nb06-prereq-gate" style="font-family:system-ui,sans-serif;margin:0 0 16px"></div>
<script>
(function(){
  var PREV_KEY='nb05_passed';
  var PREV_LABEL='Module 5: Exposure Timelines';
  var gate=document.getElementById('nb06-prereq-gate');
  if(!gate)return;
  if(!localStorage.getItem(PREV_KEY)){
    gate.innerHTML='<div style="background:#fff3cd;border:1px solid #ffc107;border-left:5px solid #ffc107;border-radius:6px;padding:14px 18px;color:#664d03"><b>&#9888; Prerequisite not detected</b><br>It looks like you have not yet passed the quiz for <b>'+PREV_LABEL+'</b>. We recommend completing that module first before continuing here. Your facilitator can help if needed.</div>';
  }else{
    gate.innerHTML='<div style="background:#d1e7dd;border-left:5px solid #198754;border-radius:6px;padding:10px 16px;color:#0a3622"><b>&#10003; '+PREV_LABEL+' completed</b> &#8212; welcome to the next module.</div>';
  }
})();
</script>

## Setting the scene — the analysis-ready dataset

Before we run a single statistical model, let us appreciate what we now hold: a row-level birth record dataset in which every birth is annotated with its own trimester climate history. Each row carries the baby's outcome (birth weight, neonatal survival), the mother's profile (age, education, wealth, urban/rural), the cluster's coordinates, and the land-surface temperature, vegetation index, and rainfall the mother was exposed to in each of her three trimesters.

This dataset is the output of five notebooks of pipeline work. In a real study you would spend weeks cleaning and validating it. Today you are holding the finished product — use it.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy import stats
import statsmodels.api as sm

import pathlib, sys

def _find_data():
    here = pathlib.Path().resolve()
    for p in (here, here.parent, here.parent.parent):
        if (p / 'data').is_dir():
            if str(p) not in sys.path:
                sys.path.insert(0, str(p))
            return p / 'data'
    raise FileNotFoundError('Cannot find data/ directory')

DATA_DIR = _find_data()
print(f'Data directory: {DATA_DIR}')

def mc_question(question, options, correct, explanation):
    try:
        import ipywidgets as w
        from IPython.display import display, HTML
        header = w.HTML(f'<div style="font-size:15px;font-weight:600;margin-bottom:10px">🧠 {question}</div>')
        radio = w.RadioButtons(options=[f"{k})  {v}" for k, v in options.items()],
                               layout=w.Layout(width="100%"),
                               style={"description_width": "0px"})
        feedback = w.Output()
        def _check(change):
            if change["name"] != "value": return
            letter = change["new"][0]
            with feedback:
                feedback.clear_output(wait=True)
                if letter == correct:
                    display(HTML(f'<div style="background:#d4edda;color:#155724;padding:10px 14px;border-radius:6px;margin-top:6px;border-left:4px solid #28a745">✅ <b>Correct!</b>&nbsp; {explanation}</div>'))
                else:
                    display(HTML(f'<div style="background:#f8d7da;color:#721c24;padding:10px 14px;border-radius:6px;margin-top:6px;border-left:4px solid #dc3545">❌ <b>Not quite.</b>&nbsp; The answer is <b>{correct}</b>. {explanation}</div>'))
        radio.observe(_check, names="value")
        display(w.VBox([header, radio, feedback],
                       layout=w.Layout(border="1px solid #dee2e6", padding="16px 20px",
                                       border_radius="8px", margin="16px 0")))
    except ImportError:
        print(f"Q: {question}")
        for k, v in options.items():
            print(f"  {k}) {v}{' ← CORRECT' if k == correct else ''}")
        print(f"Explanation: {explanation}")

import duckdb

# Load core data
# Each student's trimester file is in their home directory (written by Notebook 05).
# If it exists there we use it; otherwise we fall back to the shared pre-generated copy.
import pathlib as _pl
_home_trimester = _pl.Path.home() / 'zw_births_trimester.csv'
_shared_trimester = DATA_DIR / 'zw_births_trimester.csv'
_trimester_path = _home_trimester if _home_trimester.exists() else _shared_trimester
b         = pd.read_csv(_trimester_path)
print(f'Loaded trimester data from: {_trimester_path}')
clusters  = pd.read_csv(DATA_DIR / 'zw_clusters.csv')
climate   = pd.read_csv(DATA_DIR / 'zw_climate_monthly.csv')

# Annual mean climate per cluster — climate means only; cluster metadata stays in `clusters`
clim_ann = (
    climate
    .groupby('cluster_id')[['temp_c', 'rain_mm', 'ndvi', 'pm25', 'dust_aod550']]
    .mean()
    .reset_index()
    .rename(columns={'temp_c': 'mean_temp', 'rain_mm': 'mean_rain', 'ndvi': 'mean_ndvi',
                     'pm25': 'mean_pm25', 'dust_aod550': 'mean_dust'})
)

# Birth records with valid birth weight for regression
b_lbw = b[b['low_birth_weight'].notna()].copy()

print(f"Births loaded: {len(b):,}  |  with LBW flag: {len(b_lbw):,}")
print(f"Clusters: {len(clusters)}  |  Climate rows: {len(climate):,}")
print(f"clim_ann cols: {list(clim_ann.columns)}")

---
## Part A — Where are adverse outcomes concentrated?

Before modelling, we examine the spatial distribution of outcomes. Zimbabwe spans dramatic
gradients: the cool Eastern Highlands (Manicaland), hot dry lowveld (Masvingo, Matabeleland South),
and urban centres (Harare, ).


### Why start at the province level?

Geography is often the first thing that reveals whether an environmental hypothesis is worth pursuing. If low birth weight (LBW) were uniformly distributed across Zimbabwe, a climate-health link would be implausible — LBW has many determinants, and a flat spatial pattern suggests environment is not a major one.

Scan the table for provinces with both high LBW rates *and* extreme LST or low NDVI. Do the hot, dry lowveld provinces (Matabeleland South, Masvingo) cluster at the top? That spatial clustering is exactly what motivates the regression analysis in Part B.


### ✅ Worked example A1 — Province-level outcome table


In [ ]:
prov = b.groupby('province').agg(
    n_births         = ('neonatal_death', 'count'),
    lbw_n            = ('low_birth_weight', 'sum'),
    lbw_denom        = ('low_birth_weight', lambda x: x.notna().sum()),
    neonatal_deaths  = ('neonatal_death', 'sum'),
    mean_temp_t3     = ('temp_t3', 'mean'),
    mean_ndvi_t3     = ('ndvi_t3', 'mean'),
    pct_urban        = ('urban', 'mean'),
).reset_index()

prov['lbw_pct']         = (prov['lbw_n'] / prov['lbw_denom'] * 100).round(1)
prov['neonatal_per1000'] = (prov['neonatal_deaths'] / prov['n_births'] * 1000).round(1)

display_cols = prov[['province','lbw_pct','neonatal_per1000','mean_temp_t3','mean_ndvi_t3','pct_urban']].copy()
display_cols['pct_urban']     = (display_cols['pct_urban']*100).round(0)
display_cols['mean_temp_t3']  = display_cols['mean_temp_t3'].round(1)
display_cols['mean_ndvi_t3']  = display_cols['mean_ndvi_t3'].round(3)
display_cols.columns = ['Province','LBW %','Neonatal MR/1000','Mean T3 Temp (°C)','Mean T3 NDVI','% Urban']
print(display_cols.sort_values('LBW %', ascending=False).to_string(index=False))

### What the maps will show — and why dots matter

A choropleth by province hides within-province variation. These scatter maps plot every DHS cluster as a dot — preserving the granularity that province-level summaries erase. You will see that Zimbabwe's landscape produces a mosaic: cool green clusters in the Eastern Highlands sit alongside hot arid clusters in the south-west, often within the same province.

Look at the left panel (LBW rate) and right panel (NDVI) side by side. Do clusters with low vegetation (yellow dots, right) tend to have elevated LBW (dark red, left)? That visual impression is the ecological hypothesis — the regressions will test it formally.


### ✅ Worked example A2 — Cluster scatter maps: LBW and LST


In [ ]:
import contextily as ctx
from pyproj import Transformer as _T

cl_merged = clusters.merge(clim_ann, on='cluster_id')
cl_lbw    = cl_merged[cl_merged['lbw_denom'] >= 5].copy()

_proj = _T.from_crs("EPSG:4326", "EPSG:3857", always_xy=True)
for df in [cl_merged, cl_lbw]:
    mx, my = _proj.transform(df['cluster_lon'].values, df['cluster_lat'].values)
    df['_mx'] = mx; df['_my'] = my

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

sc1 = axes[0].scatter(cl_lbw['_mx'], cl_lbw['_my'],
    c=cl_lbw['lbw_rate']*100, cmap='YlOrRd', s=35, alpha=0.85, vmin=0, vmax=25, zorder=3)
plt.colorbar(sc1, ax=axes[0], label='LBW rate (%)')
axes[0].set_title('Cluster LBW Rate', fontsize=12, fontweight='bold')
axes[0].set_axis_off()
try: ctx.add_basemap(axes[0], source=ctx.providers.CartoDB.Positron, zoom=7)
except Exception: pass

sc2 = axes[1].scatter(cl_merged['_mx'], cl_merged['_my'],
    c=cl_merged['mean_ndvi'], cmap='YlGn', s=35, alpha=0.85, vmin=0.15, vmax=0.55, zorder=3)
plt.colorbar(sc2, ax=axes[1], label='Mean NDVI')
axes[1].set_title('Mean NDVI 2011–2015', fontsize=12, fontweight='bold')
axes[1].set_axis_off()
try: ctx.add_basemap(axes[1], source=ctx.providers.CartoDB.Positron, zoom=7)
except Exception: pass

plt.suptitle('Zimbabwe DHS 2015: Birth Outcomes and Environmental Conditions',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()
print('Visual: Do high-LBW clusters (dark red) tend to occur in lower-NDVI (yellow) areas?')

---
### 🧪 Exercise 1 — Cluster-level ecological correlations *(3 points)*

Using only clusters with `lbw_denom >= 5`, compute the Pearson correlation between:
1. Mean annual NDVI and cluster LBW rate → store as `r_ndvi_lbw` (float, 3 dp)
2. Mean annual LST and cluster LBW rate → store as `r_lst_lbw` (float, 3 dp)

Plot both as scatter plots with regression lines.


In [ ]:
cl_eco = cl_merged[cl_merged['lbw_denom'] >= 5].copy()

r_ndvi_lbw_raw, p_ndvi = stats.pearsonr(cl_eco['mean_ndvi'], cl_eco['lbw_rate'])
r_temp_lbw_raw, p_temp  = stats.pearsonr(cl_eco['mean_temp'],  cl_eco['lbw_rate'])
r_ndvi_lbw = round(r_ndvi_lbw_raw, 3)
r_temp_lbw  = round(r_temp_lbw_raw,  3)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, xcol, r_val, p_val, xlabel in [
    (axes[0], 'mean_ndvi',  r_ndvi_lbw, p_ndvi, 'Mean annual NDVI'),
    (axes[1], 'mean_temp',  r_temp_lbw, p_temp, 'Mean annual temperature (°C)'),
]:
    x = cl_eco[xcol]; y = cl_eco['lbw_rate'] * 100
    ax.scatter(x, y, alpha=0.4, s=15, color='#2a9d8f')
    m, b_ = np.polyfit(x, y, 1)
    xline = np.linspace(x.min(), x.max(), 100)
    ax.plot(xline, m*xline+b_, 'r-', lw=2)
    ax.set_xlabel(xlabel); ax.set_ylabel('Cluster LBW rate (%)')
    ax.set_title(f'r = {r_val:.3f}  (p = {p_val:.3f})', fontsize=11)

plt.suptitle('Cluster-level Environmental Correlates of Low Birth Weight', fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()

print(f'r(NDVI, LBW rate)  = {r_ndvi_lbw}  (p={p_ndvi:.3f})')
print(f'r(Temp, LBW rate)  = {r_temp_lbw}   (p={p_temp:.3f})')

In [ ]:
%%html
<div style="font-family:system-ui,sans-serif;border:1px solid #dee2e6;border-radius:8px;padding:16px 20px;margin:16px 0;background:#fafafa">
  <div style="font-size:15px;font-weight:600;margin-bottom:12px">🧠 The regression shows higher T3 LST is associated with increased LBW odds. What alternative explanation (confounding) should you always consider?</div>
  <div style="margin:7px 0"><label style="cursor:pointer;display:flex;align-items:flex-start;gap:10px"><input type="radio" name="nb06q1" value="A" style="margin-top:3px;flex-shrink:0"><span><b>A)</b> The regression model may have a coding error</span></label></div>
  <div style="margin:7px 0"><label style="cursor:pointer;display:flex;align-items:flex-start;gap:10px"><input type="radio" name="nb06q1" value="B" style="margin-top:3px;flex-shrink:0"><span><b>B)</b> Hot, arid areas also tend to have higher poverty and food insecurity, which independently increase LBW risk — heat and poverty are co-located</span></label></div>
  <div style="margin:7px 0"><label style="cursor:pointer;display:flex;align-items:flex-start;gap:10px"><input type="radio" name="nb06q1" value="C" style="margin-top:3px;flex-shrink:0"><span><b>C)</b> LST directly causes LBW with no mediating pathway</span></label></div>
  <div style="margin:7px 0"><label style="cursor:pointer;display:flex;align-items:flex-start;gap:10px"><input type="radio" name="nb06q1" value="D" style="margin-top:3px;flex-shrink:0"><span><b>D)</b> The association is only present in urban clusters</span></label></div>
  <div id="nb06q1-fb" style="margin-top:10px"></div>
</div>
<script>
(function(){
  var C="B",E="This is the classic confounding problem in ecological studies. Temperature and socioeconomic deprivation are spatially correlated in Zimbabwe. Controlling for wealth index partially addresses this, but residual confounding is always possible.",ID="nb06q1";
  function showFeedback(val){
    var fb=document.getElementById(ID+"-fb");
    if(!fb)return;
    if(val===C){
      fb.innerHTML='<div style="background:#d4edda;color:#155724;padding:10px 14px;border-radius:6px;border-left:4px solid #28a745">✅ <b>Correct!</b> '+E+'</div>';
    }else{
      fb.innerHTML='<div style="background:#f8d7da;color:#721c24;padding:10px 14px;border-radius:6px;border-left:4px solid #dc3545">❌ <b>Not quite.</b> The answer is <b>'+C+'</b>. '+E+'</div>';
    }
    document.dispatchEvent(new CustomEvent('mcq-update'));
  }
  function attach(){
    var saved=localStorage.getItem('mcq_'+ID);
    document.querySelectorAll('input[name="'+ID+'"]').forEach(function(r){
      if(saved&&r.value===saved){r.checked=true;}
      r.addEventListener("change",function(){
        localStorage.setItem('mcq_'+ID,this.value);
        showFeedback(this.value);
      });
    });
    if(saved)showFeedback(saved);
  }
  if(document.readyState==="loading"){document.addEventListener("DOMContentLoaded",attach);}else{attach();}
})();
</script>

---
## Part B — Individual-level logistic regression

We now move from ecological (cluster-level) to individual-level analysis.
**Outcome**: low birth weight (< 2,500 g). **Predictors**: trimester-specific
environmental exposures + socioeconomic and maternal factors.

> An **odds ratio (OR)** of 1.0 = no association. OR > 1 = higher odds of LBW.
> OR < 1 = lower odds. The 95% CI tells us the range of plausible values.
> If the CI crosses 1.0, the association is not statistically significant at α = 0.05.


### Why logistic regression and how to read the output

Low birth weight is a **binary outcome** (< 2,500 g or not), so we use logistic regression — the standard epidemiological workhorse for binary outcomes. The model simultaneously adjusts for multiple predictors, which is essential here: temperature and poverty are spatially co-located in Zimbabwe, so we must "hold wealth constant" before asking what heat adds.

**Reading an odds ratio (OR):** OR = 1.0 means no association. OR = 1.3 means 30% higher odds of LBW for a one-unit increase in the predictor. OR = 0.8 means 20% lower odds. The 95% confidence interval (CI) captures uncertainty; if the CI crosses 1.0, the result is not statistically significant at α = 0.05.

**Predictors we control for:** wealth quintile (poorer mothers have worse nutrition and healthcare), urban/rural (facility-delivery rates differ), maternal age (adolescent and older mothers have worse outcomes), birth order, and education level. These are *confounders* — variables that correlate with both the exposure (heat) and the outcome (LBW).


### 
✅ Worked example B1 — Fit the logistic regression


In [ ]:
mod_df = b_lbw.copy()
mod_df['edu_num'] = mod_df['education'].map({'none':0,'primary':1,'secondary':2,'higher':3})

# Standardise continuous predictors for comparable coefficients
for col in ['temp_t3', 'rain_t3', 'ndvi_t3', 'pm25_t3', 'dust_t3', 'mother_age']:
    mu, sd = mod_df[col].mean(), mod_df[col].std()
    mod_df[f'{col}_z'] = (mod_df[col] - mu) / sd

X_cols = ['temp_t3_z', 'rain_t3_z', 'ndvi_t3_z', 'pm25_t3_z', 'dust_t3_z',
          'urban', 'wealth_quintile', 'birth_order', 'edu_num', 'mother_age_z']
X = sm.add_constant(mod_df[X_cols].astype(float))
y = mod_df['low_birth_weight'].astype(float)

# Drop rows with any missing predictor or outcome
valid = X.notna().all(axis=1) & y.notna()
X, y = X[valid], y[valid]
print(f'Fitting on {len(y):,} births (dropped {(~valid).sum()} with missing values)')

model = sm.Logit(y, X).fit(disp=False)

odds   = np.exp(model.params[1:])
ci_lo  = np.exp(model.conf_int().iloc[1:, 0])
ci_hi  = np.exp(model.conf_int().iloc[1:, 1])
pvals  = model.pvalues[1:]

labels = [
    'T3 temperature (+1 SD)', 'T3 rainfall (+1 SD)', 'T3 NDVI (+1 SD)',
    'T3 PM2.5 (+1 SD)', 'T3 dust AOD (+1 SD)',
    'Urban cluster', 'Wealth quintile (+1)',
    'Birth order (+1)', 'Education (+1 level)', 'Maternal age (+1 SD)',
]

results = pd.DataFrame({'OR': odds.values, 'CI_lo': ci_lo.values, 'CI_hi': ci_hi.values,
                         'p': pvals.values}, index=labels)
results['Sig'] = results['p'].apply(
    lambda p: '***' if p<0.001 else ('**' if p<0.01 else ('*' if p<0.05 else '')))
results['OR']    = results['OR'].round(3)
results['CI_lo'] = results['CI_lo'].round(3)
results['CI_hi'] = results['CI_hi'].round(3)
results['p']     = results['p'].round(3)

print(results.to_string())
print(f"\nPseudo R-squared (McFadden): {model.prsquared:.3f}")

### The forest plot — reading associations at a glance

A forest plot is the epidemiologist's standard display for regression results. Each predictor is a row; the dot is the point estimate (OR) and the horizontal whiskers are the 95% CI. The dashed vertical line at OR = 1.0 is the null: dots to the left of it are protective, dots to the right indicate higher LBW odds.

Colour coding makes significance visible at a glance — teal dots (CI entirely < 1.0) are protective; red dots (CI entirely > 1.0) are risk factors; grey dots have a CI that crosses the null. When you see environmental predictors landing in grey, that is a finding too — it tells us the signal exists but the data cannot resolve it cleanly. Why? Cell 15 explains the three reasons.


### ✅ Worked example B2 — Forest plot


In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
y_pos = np.arange(len(results))

colours = ['#2a9d8f' if ch < 1 else ('#e63946' if cl > 1 else '#888888')
           for cl, ch in zip(results['CI_lo'], results['CI_hi'])]

# Draw error bars without markers, then scatter markers individually for per-point colour
ax.errorbar(
    results['OR'], y_pos,
    xerr=[results['OR']-results['CI_lo'], results['CI_hi']-results['OR']],
    fmt='none', ecolor='k', capsize=4, capthick=1.5, lw=1.2,
)
for xi, yi, c in zip(results['OR'], y_pos, colours):
    ax.plot(xi, yi, 'o', markersize=8, markerfacecolor=c, markeredgecolor='k', markeredgewidth=0.8)

ax.axvline(1.0, color='black', linestyle='--', lw=1.2)
ax.set_yticks(y_pos)
ax.set_yticklabels(results.index, fontsize=10)
ax.set_xlabel('Odds Ratio (95% CI)', fontsize=11)
ax.set_title('Predictors of Low Birth Weight — Zimbabwe DHS 2015\nLogistic regression, n = {:,}'.format(len(y)),
             fontsize=11, fontweight='bold')
ax.set_xlim(0.45, 1.85)

handles = [
    mpatches.Patch(color='#2a9d8f', label='OR < 1 (protective — CI excludes 1.0)'),
    mpatches.Patch(color='#e63946', label='OR > 1 (risk — CI excludes 1.0)'),
    mpatches.Patch(color='#888888', label='CI crosses 1.0 (inconclusive)'),
]
ax.legend(handles=handles, loc='lower right', fontsize=9)
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

> **Findings.** Significant predictors: maternal education (protective), birth order (protective),
> maternal age (slight risk). Environmental predictors (heat, rainfall, NDVI) are in the expected
> directions but confidence intervals cross 1.0.

> **Why?** Three reasons: (1) cluster jitter means exposure is imprecise;
> (2) birth weight is only recorded for ~30% of births — selection is non-random
> (facility-delivered, urban births overrepresented); (3) monthly CMC birth date gives
> only a rough proxy for trimester exposure timing, introducing noise.
> **Environmental epidemiology: study design and data quality determine what we detect.**


---
### 🧪 Exercise 2 — Interpret and extend the model *(4 points)*

**(a)** Extract the environmental predictor rows from `results` into `env_results` (DataFrame).

**(b)** Compute the **province-level** Pearson correlation between mean NDVI and LBW rate.
Store as `r_prov_ndvi_lbw` (float, 3 dp). Compare to the individual-level r.
Store `'stronger'` or `'weaker'` in `ndvi_level_comparison`.

**(c)** Write 2–3 sentences in the free-text cell below explaining why the ecological
association between NDVI and LBW is stronger than the individual-level association.


In [ ]:
env_labels = ['T3 temperature (+1 SD)', 'T3 rainfall (+1 SD)', 'T3 NDVI (+1 SD)', 'T3 PM2.5 (+1 SD)', 'T3 dust AOD (+1 SD)']
env_results = results.loc[env_labels].copy()

prov_agg = b.groupby('province').agg(
    lbw_rate  = ('low_birth_weight', 'mean'),
    mean_ndvi = ('ndvi_t3', 'mean'),
).dropna()
r_raw, _ = stats.pearsonr(prov_agg['mean_ndvi'], prov_agg['lbw_rate'])
r_prov_ndvi_lbw = round(r_raw, 3)

r_ind, _ = stats.pearsonr(b_lbw['ndvi_t3'].dropna(), b_lbw['low_birth_weight'].dropna())
ndvi_level_comparison = 'stronger' if abs(r_prov_ndvi_lbw) > abs(r_ind) else 'weaker'

print('Environmental ORs:')
print(env_results)
print(f'\nProvince NDVI_T3 ~ LBW: r = {r_prov_ndvi_lbw}')
print(f'Individual NDVI_T3 ~ LBW: r = {r_ind:.3f}')
print(f'Ecological association is {ndvi_level_comparison} than individual-level')

In [ ]:
%%html
<div style="font-family:system-ui,sans-serif;border:1px solid #dee2e6;border-radius:8px;padding:16px 20px;margin:16px 0;background:#fafafa">
  <div style="font-size:15px;font-weight:600;margin-bottom:12px">🧠 The GeoAI assistant shows you the SQL it generated before running it. Why is this transparency important?</div>
  <div style="margin:7px 0"><label style="cursor:pointer;display:flex;align-items:flex-start;gap:10px"><input type="radio" name="nb06q2" value="A" style="margin-top:3px;flex-shrink:0"><span><b>A)</b> Databases require human approval before executing any query</span></label></div>
  <div style="margin:7px 0"><label style="cursor:pointer;display:flex;align-items:flex-start;gap:10px"><input type="radio" name="nb06q2" value="B" style="margin-top:3px;flex-shrink:0"><span><b>B)</b> Showing the SQL lets you verify that the query actually answers the question you asked — AI can misinterpret ambiguous natural language</span></label></div>
  <div style="margin:7px 0"><label style="cursor:pointer;display:flex;align-items:flex-start;gap:10px"><input type="radio" name="nb06q2" value="C" style="margin-top:3px;flex-shrink:0"><span><b>C)</b> SQL is decorative and helps the interface look more technical</span></label></div>
  <div style="margin:7px 0"><label style="cursor:pointer;display:flex;align-items:flex-start;gap:10px"><input type="radio" name="nb06q2" value="D" style="margin-top:3px;flex-shrink:0"><span><b>D)</b> Transparency is required by GDPR</span></label></div>
  <div id="nb06q2-fb" style="margin-top:10px"></div>
</div>
<script>
(function(){
  var C="B",E="AI-generated SQL should always be treated as a draft. Reviewing it closes the loop between your intent and the computer\'s interpretation — a critical skill as AI tools become routine in research workflows.",ID="nb06q2";
  function showFeedback(val){
    var fb=document.getElementById(ID+"-fb");
    if(!fb)return;
    if(val===C){
      fb.innerHTML='<div style="background:#d4edda;color:#155724;padding:10px 14px;border-radius:6px;border-left:4px solid #28a745">✅ <b>Correct!</b> '+E+'</div>';
    }else{
      fb.innerHTML='<div style="background:#f8d7da;color:#721c24;padding:10px 14px;border-radius:6px;border-left:4px solid #dc3545">❌ <b>Not quite.</b> The answer is <b>'+C+'</b>. '+E+'</div>';
    }
    document.dispatchEvent(new CustomEvent('mcq-update'));
  }
  function attach(){
    var saved=localStorage.getItem('mcq_'+ID);
    document.querySelectorAll('input[name="'+ID+'"]').forEach(function(r){
      if(saved&&r.value===saved){r.checked=true;}
      r.addEventListener("change",function(){
        localStorage.setItem('mcq_'+ID,this.value);
        showFeedback(this.value);
      });
    });
    if(saved)showFeedback(saved);
  }
  if(document.readyState==="loading"){document.addEventListener("DOMContentLoaded",attach);}else{attach();}
})();
</script>

*(Write 2–3 sentences: why is the ecological NDVI–LBW association stronger than the individual-level one?)*


## Part C preview — from space to time

Parts A and B examined *where* adverse outcomes concentrate. Part C asks *when*. If climate drives LBW, we expect seasonal patterns: births after the hottest trimester of pregnancy (late dry season, August–October) should have higher LBW rates than births after cooler, wetter trimesters.

Seasonality is a natural experiment of sorts — the same cluster experiences both wet and dry seasons year after year. Any systematic shift in LBW rates by birth month is ecologically meaningful, even though it does not prove causation.


---
## Part C — Seasonality and exposure categories


### Dichotomising environment — a complementary lens

While continuous correlations (Part A) and regression coefficients (Part B) are the analytical gold standard, dichotomising clusters into "hot vs cool" and "green vs sparse" gives an immediately intuitive comparison. Think of it as a sanity check: if the regression odds ratios point towards heat increasing LBW, the bar chart should show higher LBW in the hot-cluster group.

A word of caution: dichotomisation discards information and inflates the apparent effect size in some situations. Use these bar charts for communication and hypothesis illustration — not as the primary analysis.


### ✅ Worked example C1 — LBW rates by environmental category


In [ ]:
cl_cat = cl_merged[['cluster_id','mean_temp','mean_rain','mean_ndvi']].copy()
cl_cat['hot_cluster']   = cl_cat['mean_temp'] > cl_cat['mean_temp'].median()
cl_cat['green_cluster'] = cl_cat['mean_ndvi'] > cl_cat['mean_ndvi'].median()
b3 = b.merge(cl_cat, on='cluster_id')

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
for ax, col, cats, colours, title in [
    (axes[0], 'hot_cluster',   [False, True],  ['#457b9d','#e63946'], 'Temperature category'),
    (axes[1], 'green_cluster', [False, True],  ['#e9c46a','#2a9d8f'], 'Vegetation category'),
]:
    rates = b3[b3['low_birth_weight'].notna()].groupby(col)['low_birth_weight'].mean() * 100
    xlabels = ['Cool','Hot'] if col=='hot_cluster' else ['Sparse','Green']
    bars = ax.bar(xlabels, [rates.get(c, 0) for c in cats], color=colours, width=0.5)
    for bar, val in zip(bars, [rates.get(c,0) for c in cats]):
        ax.text(bar.get_x()+bar.get_width()/2, val+0.2, f'{val:.1f}%', ha='center', va='bottom', fontsize=12)
    ax.set_ylabel('LBW rate (%)'); ax.set_xlabel(title); ax.set_ylim(0, 13)

plt.suptitle('LBW Rate by Cluster Environmental Category', fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()

---
### 🧪 Exercise 3 — Seasonality of LBW *(3 points)*

Compute the LBW rate (%) by **birth month** (1=Jan, …, 12=Dec).
Store as `monthly_lbw` (Series, index=birth_month). Plot with a twin-axis showing mean monthly LST.
Store:
- `peak_lbw_month` — int: the birth month with the highest LBW rate
- `peak_lbw_season` — string: 'rainy' or 'dry' (rainy season = months 10, 11, 12, 1, 2, 3, 4)


In [ ]:
monthly_lbw  = (b_lbw.groupby('birth_month')['low_birth_weight'].mean() * 100).round(2)
peak_lbw_month  = int(monthly_lbw.idxmax())
rainy_months = {10,11,12,1,2,3,4}
peak_lbw_season = 'rainy' if peak_lbw_month in rainy_months else 'dry'

monthly_temp = climate.groupby('month')['temp_c'].mean()

fig, ax1 = plt.subplots(figsize=(10, 5))
ax2 = ax1.twinx()
months = list(range(1,13))
ax1.bar(months, [monthly_lbw.get(m, np.nan) for m in months], color='#e63946', alpha=0.7, label='LBW rate (%)')
ax2.plot(months, [monthly_temp.get(m, np.nan) for m in months], 'b-o', lw=2, ms=6, label='Mean temp (°C)')
for m in rainy_months:
    ax1.axvspan(m-0.5, m+0.5, alpha=0.1, color='#457b9d')
ax1.set_xticks(months)
ax1.set_xticklabels(['J','F','M','A','M','J','J','A','S','O','N','D'])
ax1.set_xlabel('Birth month'); ax1.set_ylabel('LBW rate (%)', color='#e63946')
ax2.set_ylabel('Mean 2 m air temp (°C)', color='blue')
ax1.set_title('Monthly LBW rate and mean temperature (blue shading = rainy season)', fontweight='bold')
l1,lb1 = ax1.get_legend_handles_labels()
l2,lb2 = ax2.get_legend_handles_labels()
ax1.legend(l1+l2, lb1+lb2, loc='upper left')
plt.tight_layout(); plt.show()

print(f'Peak LBW birth month: {peak_lbw_month} ({peak_lbw_season} season)')

In [ ]:
%%html
<div style="font-family:system-ui,sans-serif;border:1px solid #dee2e6;border-radius:8px;padding:16px 20px;margin:16px 0;background:#fafafa">
  <div style="font-size:15px;font-weight:600;margin-bottom:12px">🧠 You have now run the complete GeoAI pipeline. What is the most important caveat about the conclusions you can draw from this analysis?</div>
  <div style="margin:7px 0"><label style="cursor:pointer;display:flex;align-items:flex-start;gap:10px"><input type="radio" name="nb06q3" value="A" style="margin-top:3px;flex-shrink:0"><span><b>A)</b> The Zimbabwe DHS dataset is too small for any conclusions</span></label></div>
  <div style="margin:7px 0"><label style="cursor:pointer;display:flex;align-items:flex-start;gap:10px"><input type="radio" name="nb06q3" value="B" style="margin-top:3px;flex-shrink:0"><span><b>B)</b> These are observational ecological associations — we cannot claim that climate directly causes LBW without ruling out confounders and without individual-level exposure data</span></label></div>
  <div style="margin:7px 0"><label style="cursor:pointer;display:flex;align-items:flex-start;gap:10px"><input type="radio" name="nb06q3" value="C" style="margin-top:3px;flex-shrink:0"><span><b>C)</b> Logistic regression is not valid for birth outcome data</span></label></div>
  <div style="margin:7px 0"><label style="cursor:pointer;display:flex;align-items:flex-start;gap:10px"><input type="radio" name="nb06q3" value="D" style="margin-top:3px;flex-shrink:0"><span><b>D)</b> DHS data is not peer-reviewed and cannot be cited</span></label></div>
  <div id="nb06q3-fb" style="margin-top:10px"></div>
</div>
<script>
(function(){
  var C="B",E="Ecological studies identify patterns and generate hypotheses. Moving from association to causation requires individual-level data, longitudinal study designs, and rigorous confounder control.",ID="nb06q3";
  function showFeedback(val){
    var fb=document.getElementById(ID+"-fb");
    if(!fb)return;
    if(val===C){
      fb.innerHTML='<div style="background:#d4edda;color:#155724;padding:10px 14px;border-radius:6px;border-left:4px solid #28a745">✅ <b>Correct!</b> '+E+'</div>';
    }else{
      fb.innerHTML='<div style="background:#f8d7da;color:#721c24;padding:10px 14px;border-radius:6px;border-left:4px solid #dc3545">❌ <b>Not quite.</b> The answer is <b>'+C+'</b>. '+E+'</div>';
    }
    document.dispatchEvent(new CustomEvent('mcq-update'));
  }
  function attach(){
    var saved=localStorage.getItem('mcq_'+ID);
    document.querySelectorAll('input[name="'+ID+'"]').forEach(function(r){
      if(saved&&r.value===saved){r.checked=true;}
      r.addEventListener("change",function(){
        localStorage.setItem('mcq_'+ID,this.value);
        showFeedback(this.value);
      });
    });
    if(saved)showFeedback(saved);
  }
  if(document.readyState==="loading"){document.addEventListener("DOMContentLoaded",attach);}else{attach();}
})();
</script>

## Part D preview — from code to conversation

Everything you have done so far required writing Python. Part D flips the interface: you ask the database questions in plain English and a large language model translates your words into SQL. This is the GeoAI layer of the workshop — and it is where Earth observation professionals and public health researchers who are not SQL experts can still interrogate the dataset.

But first, we demonstrate raw DuckDB SQL directly, so you understand *what* the AI is generating on your behalf. Understanding the SQL is non-negotiable — you cannot evaluate an AI-generated answer if you cannot read the query that produced it.


---
## Part D — DuckDB & GeoAI: Querying the epidemiological database

The workshop database `exposures_zw.duckdb` contains all tables. We can use SQL directly,
or use the **GeoAI** natural-language interface to generate queries from plain English.


### SQL directly on the epidemiological database

`exposures_zw.duckdb` is a portable, self-contained analytical database — no server required. DuckDB's in-process model means you can query it from a Jupyter notebook at memory speed. The `births` table contains one row per birth with all trimester exposures already attached.

The two queries below answer questions that health programme managers would recognise immediately: which provinces carry the highest LBW burden, and does drought exposure at the time of pregnancy change outcomes? Run them, read the output, and form your own interpretation before looking at the GeoAI version of the same questions.


### ✅ Worked example D1 — SQL queries on the database


In [ ]:
DB_PATH = str(DATA_DIR / 'exposures_zw.duckdb')
con = duckdb.connect(DB_PATH, read_only=True)
print('Tables:', con.execute('SHOW TABLES').fetchdf()['name'].tolist())
print()

# Province LBW rates with environmental context
q1 = con.execute("""
    SELECT province,
           COUNT(*)                   AS n_births,
           ROUND(100.0 * SUM(low_birth_weight)
                 / NULLIF(SUM(CASE WHEN low_birth_weight IS NOT NULL THEN 1 END),0), 1) AS lbw_pct,
           ROUND(AVG((temp_t1 + temp_t2 + temp_t3) / 3.0), 1)    AS avg_temp_c,
           ROUND(AVG((ndvi_t1 + ndvi_t2 + ndvi_t3) / 3.0), 3)    AS avg_ndvi
    FROM births
    GROUP BY province ORDER BY lbw_pct DESC
""").fetchdf()
print(q1.to_string(index=False))

In [ ]:
# Compare births by third-trimester rainfall quartile (Q1 = lowest rainfall = drought-exposed)
q2 = con.execute("""
    WITH rain_quartile AS (
        SELECT *,
               NTILE(4) OVER (ORDER BY rain_t3) AS rain_t3_quartile
        FROM births
    )
    SELECT
        CASE WHEN rain_t3_quartile = 1 THEN 'Q1 — lowest T3 rainfall (drought-exposed)'
             ELSE 'Q2–Q4 — normal / high T3 rainfall' END AS rainfall_group,
        COUNT(*) AS n,
        ROUND(100.0 * SUM(low_birth_weight)
              / NULLIF(SUM(CASE WHEN low_birth_weight IS NOT NULL THEN 1 END), 0), 1) AS lbw_pct,
        ROUND(AVG(birth_weight_g), 0) AS mean_bw_g,
        ROUND(1000.0 * SUM(neonatal_death) / COUNT(*), 1) AS neonatal_mr
    FROM rain_quartile
    GROUP BY rainfall_group
    ORDER BY rainfall_group
""").fetchdf()
print("T3 rainfall quartile comparison:")
print(q2.to_string(index=False))


---
### 🧪 Exercise 4 — Write your own SQL query *(2 points)*

Write a SQL query returning the **top 3 provinces by neonatal mortality rate** (per 1,000 births),
with columns for n_births, neonatal_deaths, and the rate. Store as `top_nm_provinces`.


In [ ]:
top_nm_provinces = con.execute("""
    SELECT province,
           COUNT(*) AS n_births,
           SUM(neonatal_death) AS neonatal_deaths,
           ROUND(1000.0 * SUM(neonatal_death) / COUNT(*), 1) AS neonatal_mr_per1000
    FROM births
    GROUP BY province
    ORDER BY neonatal_mr_per1000 DESC
    LIMIT 3
""").fetchdf()
print(top_nm_provinces)


In [ ]:
%%html
<div style="font-family:system-ui,sans-serif;border:1px solid #dee2e6;border-radius:8px;padding:16px 20px;margin:16px 0;background:#fafafa">
  <div style="font-size:15px;font-weight:600;margin-bottom:12px">🧠 Thinking about the full pipeline you have completed — where is the role of remote sensing professionals most critical?</div>
  <div style="margin:7px 0"><label style="cursor:pointer;display:flex;align-items:flex-start;gap:10px"><input type="radio" name="nb06q4" value="A" style="margin-top:3px;flex-shrink:0"><span><b>A)</b> Only in Notebook 05 when we discuss Earth Engine</span></label></div>
  <div style="margin:7px 0"><label style="cursor:pointer;display:flex;align-items:flex-start;gap:10px"><input type="radio" name="nb06q4" value="B" style="margin-top:3px;flex-shrink:0"><span><b>B)</b> Nowhere — remote sensing is background infrastructure</span></label></div>
  <div style="margin:7px 0"><label style="cursor:pointer;display:flex;align-items:flex-start;gap:10px"><input type="radio" name="nb06q4" value="C" style="margin-top:3px;flex-shrink:0"><span><b>C)</b> Throughout: selecting appropriate EO products, understanding spatial resolution and temporal coverage, validating satellite-derived estimates, and communicating what the data can and cannot show</span></label></div>
  <div style="margin:7px 0"><label style="cursor:pointer;display:flex;align-items:flex-start;gap:10px"><input type="radio" name="nb06q4" value="D" style="margin-top:3px;flex-shrink:0"><span><b>D)</b> Only when producing the final maps in Notebook 06</span></label></div>
  <div id="nb06q4-fb" style="margin-top:10px"></div>
</div>
<script>
(function(){
  var C="C",E="Remote sensing expertise is embedded in every stage: knowing which product to use (NB02), understanding resolution limitations (NB05), interpreting spatial patterns (NB03, NB06). This course bridges EO and health — both communities need each other.",ID="nb06q4";
  function showFeedback(val){
    var fb=document.getElementById(ID+"-fb");
    if(!fb)return;
    if(val===C){
      fb.innerHTML='<div style="background:#d4edda;color:#155724;padding:10px 14px;border-radius:6px;border-left:4px solid #28a745">✅ <b>Correct!</b> '+E+'</div>';
    }else{
      fb.innerHTML='<div style="background:#f8d7da;color:#721c24;padding:10px 14px;border-radius:6px;border-left:4px solid #dc3545">❌ <b>Not quite.</b> The answer is <b>'+C+'</b>. '+E+'</div>';
    }
    document.dispatchEvent(new CustomEvent('mcq-update'));
  }
  function attach(){
    var saved=localStorage.getItem('mcq_'+ID);
    document.querySelectorAll('input[name="'+ID+'"]').forEach(function(r){
      if(saved&&r.value===saved){r.checked=true;}
      r.addEventListener("change",function(){
        localStorage.setItem('mcq_'+ID,this.value);
        showFeedback(this.value);
      });
    });
    if(saved)showFeedback(saved);
  }
  if(document.readyState==="loading"){document.addEventListener("DOMContentLoaded",attach);}else{attach();}
})();
</script>

---
## 🤖 GeoAI — Ask anything. Get an answer. Keep the conversation going.

The GeoAI assistant translates plain-English questions into instant results — charts, tables,
regressions — using the real Zimbabwe DHS data. It works like a knowledgeable research
collaborator: ask a question, see the result, then follow up naturally.

**Privacy is built in.** Individual participant identifiers are stripped server-side before any
analysis runs. You can ask about birth outcomes, exposures, or patterns — you will never see
or be given individual records.

**Conversation memory.** Each `ask()` call remembers the previous exchange, so you can build
analyses step by step without repeating yourself:

```python
ask("Plot LBW rate by province")
ask("Now break that down by urban vs rural")
ask("Which province has the biggest urban–rural gap?")
ask("Run a logistic regression adjusting for wealth and education")
ask("", new=True)   # start a fresh conversation
```

**What it can do:**
- Bar charts, scatter plots, maps, dual-axis charts
- Trimester exposure comparisons (T3 heat, drought, NDVI vs birth outcomes)
- Logistic regression with confounders — odds ratios and forest plots
- Seasonal patterns, province-level breakdowns, subgroup comparisons
- Follow-up and refinement of any previous analysis


### How `geoai()` works (SQL path)

`geoai()` sends your question to the server where an AI model writes a SQL query against
the Zimbabwe DHS epidemiological database, executes it, and returns the table.
The SQL is always shown so you can verify exactly what was computed.


In [ ]:
import urllib.request as _ureq, json as _json, os as _os

GEOAI_URL = _os.environ.get('GEOAI_API_URL', 'http://localhost:5000/api/workshop/geoai')

def geoai(question: str) -> None:
    """Send a plain-English question to the GeoAI endpoint."""
    payload = _json.dumps({'question': question, 'db': 'exposures_zw'}).encode()
    try:
        req = _ureq.Request(GEOAI_URL, data=payload,
                            headers={'Content-Type': 'application/json'}, method='POST')
        with _ureq.urlopen(req, timeout=30) as resp:
            res = _json.loads(resp.read())
        print('SQL generated:')
        print(res.get('sql', '(none)'))
        df = pd.DataFrame(res.get('rows', []), columns=res.get('columns', []))
        print('\nResults:')
        print(df.to_string(index=False) if len(df) > 0 else '(no rows)')
    except Exception as e:
        print(f'GeoAI unavailable ({e}). In a connected session this returns AI-generated SQL and results.')

geoai('Which 5 clusters have the highest LBW rate among rural clusters with at least 10 weighed births?')


In [ ]:
# Try your own question here
geoai('Compare mean T3 temperature and T3 NDVI for LBW versus normal-weight births')


### The power is in the follow-up

Once you have a result, you can keep going. The conversation carries full context:

```python
ask("Plot LBW rate by province")
ask("Now split each bar by urban vs rural")            # follows on
ask("Which province has the worst rural outcome?")     # builds further
ask("Run a logistic regression for that province")     # still in context
ask("", new=True)                                      # fresh start
```

This is the GeoAI experience — not isolated queries but an evolving conversation with a
collaborator who knows the dataset. Each result is a stepping stone to the next question.


---
## Start your GeoAI conversation

Run the first cell below to get a result, then keep asking follow-up questions in the cells that follow.
Call `ask("", new=True)` to start a completely different analysis thread.


In [ ]:
import urllib.request as _ureq, json as _json, os as _os
import base64
from IPython.display import display, Image, HTML

GEOAI_ANALYZE_URL = _os.environ.get('GEOAI_ANALYZE_URL', 'http://localhost:5000/api/workshop/geoai_analyze')

# Conversation history — persists across ask() calls in this notebook session
_geoai_history = []

def ask(question: str, new: bool = False) -> None:
    """
    Ask a climate-health question in plain English.
    Results appear immediately (chart first, then summary text).
    Generated code is collapsed at the bottom for reference.

    Pass new=True to reset the conversation and start fresh.

    Examples:
        ask("Plot LBW rate by province as a bar chart")
        ask("Now break that down by urban vs rural")          # follow-up
        ask("Which province has the biggest urban–rural gap?")
        ask("", new=True)                                     # fresh start
    """
    global _geoai_history

    if new:
        _geoai_history = []
        print("── New conversation started ──")
        if not question.strip():
            return

    if not question.strip():
        print('Tip: ask("your question here")  |  ask("", new=True) to reset')
        return

    payload = _json.dumps({
        'question': question,
        'history':  _geoai_history,
    }).encode()

    try:
        req = _ureq.Request(GEOAI_ANALYZE_URL, data=payload,
                            headers={'Content-Type': 'application/json'}, method='POST')
        with _ureq.urlopen(req, timeout=60) as resp:
            res = _json.loads(resp.read())
    except Exception as e:
        print(f'GeoAI unavailable: {e}')
        return

    if res.get('error') and not res.get('figures') and not res.get('output'):
        print(f'Error: {res["error"]}')
        return

    # Update conversation history so the next ask() has context
    _geoai_history = res.get('history', _geoai_history)

    # ── Results first: figures → printed output ──────────────────────────────
    for fig_b64 in res.get('figures', []):
        display(Image(data=base64.b64decode(fig_b64)))
    if res.get('output'):
        print(res['output'].rstrip())
    if res.get('error'):
        print(f'Note: {res["error"]}')

    # ── Generated code — collapsed, for those who want to dig deeper ─────────
    code = (res.get('code') or '').strip()
    if code:
        escaped = code.replace('&','&amp;').replace('<','&lt;').replace('>','&gt;')
        display(HTML(
            '<details style="margin-top:10px;font-size:12px;color:#6c757d">'
            '<summary style="cursor:pointer;user-select:none">'
            '&#9656; View generated code</summary>'
            '<pre style="background:#f6f8fa;padding:12px 16px;border-radius:6px;'
            'overflow-x:auto;margin-top:6px;font-size:11px;line-height:1.5">'
            + escaped + '</pre></details>'
        ))


In [ ]:
# ── Run this first ─────────────────────────────────────────────────────────
ask("Plot LBW rate by province as a horizontal bar chart sorted highest to lowest")


In [ ]:
# ── Follow-up — ask() remembers the previous answer ────────────────────────
ask("Now break that down by urban vs rural for each province")


In [ ]:
# ── Another follow-up — keep building the story ────────────────────────────
ask("Which province has the biggest urban–rural gap in LBW rate?")


In [ ]:
# ── Fresh conversation — ask("", new=True) resets memory ───────────────────
ask("", new=True)
ask("Run a logistic regression of LBW on T3 temperature, drought exposure, wealth quintile, and urban — show odds ratios as a forest plot")


In [ ]:
# ── Your turn — replace this with your own question ─────────────────────────
ask("Compare mean T3 temperature for LBW vs normal-weight births by province")


---
## 🤖 Part E — GeoAI Chat: Explore freely

The cells above showed how `ask()` works under the hood — you saw the API call, the conversation history, the generated code. Now the scaffolding comes off.

The chat interface below is the same engine, but with a proper conversational UI. Type a question, hit **Send ↑** (or press Enter), and the assistant responds with charts, tables, or regression output — inline, in the notebook.

Each answer remembers the previous exchange. Ask a follow-up. Change direction. Press **+ New** to start a completely fresh thread.

> This is what GeoAI-assisted environmental epidemiology looks like in practice — a collaborator who knows the Zimbabwe DHS dataset and can turn your questions into analysis in seconds.

In [ ]:
import ipywidgets as _w
from IPython.display import display as _disp, Image as _Img, HTML as _HTML
import json as _json, base64 as _b64, urllib.request as _ureq, os as _os

def make_geoai_chat():
    _url  = _os.environ.get('GEOAI_ANALYZE_URL', 'http://localhost:5000/api/workshop/geoai_analyze')
    _hist = []

    header = _w.HTML("""
<div style="background:linear-gradient(135deg,#1b7a4b 0%,#1565c0 100%);
            color:white;padding:13px 18px;border-radius:8px 8px 0 0;
            font-family:system-ui,sans-serif;display:flex;align-items:center;gap:10px">
  <span style="font-size:22px">🤖</span>
  <div>
    <div style="font-weight:700;font-size:14px;letter-spacing:.01em">GeoAI Research Assistant</div>
    <div style="font-size:11px;opacity:.85">Zimbabwe DHS 2015 · births · climate · exposures</div>
  </div>
</div>""")

    chat = _w.Output(layout=_w.Layout(
        min_height='360px', max_height='520px',
        overflow_y='auto', padding='14px 16px',
        background_color='#ffffff',
    ))
    status = _w.Output(layout=_w.Layout(min_height='24px'))

    q_input = _w.Text(
        placeholder='Ask about births, exposures, provinces, regressions…',
        layout=_w.Layout(flex='1', min_width='0', height='34px'),
    )
    send_btn = _w.Button(
        description='Send ↑', button_style='success',
        layout=_w.Layout(width='82px', height='34px'),
    )
    new_btn = _w.Button(
        description='+ New', tooltip='Start a fresh conversation',
        layout=_w.Layout(width='68px', height='34px'),
    )
    footer = _w.HBox(
        [q_input, send_btn, new_btn],
        layout=_w.Layout(padding='8px 12px', gap='6px',
                         border_top='1px solid #dee2e6',
                         background_color='#f8f9fa'),
    )

    def _user_bubble(text):
        return (
            '<div style="text-align:right;margin:10px 0;font-family:system-ui,sans-serif">'
            f'<span style="background:#0d6efd;color:#fff;padding:8px 14px;'
            'border-radius:18px 18px 4px 18px;display:inline-block;'
            f'max-width:82%;text-align:left;font-size:13px">{text}</span></div>'
        )

    def _bot_bubble(text):
        return (
            '<div style="margin:10px 0;font-family:system-ui,sans-serif">'
            '<div style="display:flex;align-items:flex-start;gap:8px">'
            '<span style="font-size:18px;flex-shrink:0;margin-top:2px">🤖</span>'
            '<div style="background:#f1f3f5;padding:10px 14px;'
            'border-radius:4px 18px 18px 18px;max-width:85%;'
            f'font-size:13px;line-height:1.55">{text}</div>'
            '</div></div>'
        )

    def _code_block(code):
        esc = code.replace('&', '&amp;').replace('<', '&lt;').replace('>', '&gt;')
        return (
            '<details style="margin:4px 0 6px 26px;font-size:11px;'
            'color:#6c757d;font-family:system-ui,sans-serif">'
            '<summary style="cursor:pointer;user-select:none">▶ View generated code</summary>'
            '<pre style="background:#f6f8fa;padding:10px 12px;border-radius:6px;'
            f'overflow-x:auto;font-size:10.5px;line-height:1.45;margin-top:4px">{esc}</pre>'
            '</details>'
        )

    with chat:
        _disp(_HTML(
            '<div style="color:#495057;font-size:13px;font-family:system-ui,sans-serif;padding:2px 0">'
            '<b>Ask me anything about the Zimbabwe DHS 2015 data.</b><br><br>'
            'Try these starters — or go straight to your own question:<br>'
            '<span style="color:#0d6efd">· "Plot LBW rate by province sorted highest to lowest"</span><br>'
            '<span style="color:#0d6efd">· "Compare T3 temperature for LBW vs normal-weight births"</span><br>'
            '<span style="color:#0d6efd">· "Run a logistic regression of LBW on T3 temperature and wealth"</span><br>'
            '<span style="color:#6c757d;font-size:12px">'
            'Each answer remembers the previous — ask follow-ups naturally.</span>'
            '</div>'
        ))

    def _send(_=None):
        q = q_input.value.strip()
        if not q:
            return
        q_input.value = ''
        send_btn.disabled = True

        with chat:
            _disp(_HTML(_user_bubble(q)))
        with status:
            status.clear_output(wait=True)
            _disp(_HTML(
                '<div style="color:#6c757d;font-size:12px;font-family:system-ui,sans-serif;'
                'padding:2px 12px">🤖 <i>thinking…</i></div>'
            ))

        payload = _json.dumps({'question': q, 'history': _hist}).encode()
        try:
            req = _ureq.Request(_url, data=payload,
                                headers={'Content-Type': 'application/json'}, method='POST')
            with _ureq.urlopen(req, timeout=60) as resp:
                res = _json.loads(resp.read())

            _hist.clear()
            _hist.extend(res.get('history', []))
            status.clear_output()

            txt = (res.get('output') or '').strip()
            with chat:
                if txt:
                    _disp(_HTML(_bot_bubble(txt)))
                for fig in res.get('figures', []):
                    _disp(_Img(data=_b64.b64decode(fig)))
                code = (res.get('code') or '').strip()
                if code:
                    _disp(_HTML(_code_block(code)))
                if res.get('error') and not txt and not res.get('figures'):
                    _disp(_HTML(_bot_bubble(f'⚠️ {res["error"]}')))
        except Exception as e:
            status.clear_output()
            with chat:
                _disp(_HTML(_bot_bubble(f'⚠️ Could not reach GeoAI: {e}')))
        finally:
            send_btn.disabled = False

    def _new(_):
        _hist.clear()
        chat.clear_output()
        status.clear_output()
        with chat:
            _disp(_HTML(
                '<div style="text-align:center;color:#6c757d;font-size:12px;'
                'font-family:system-ui,sans-serif;padding:6px 0">'
                '── New conversation ──</div>'
            ))

    send_btn.on_click(_send)
    new_btn.on_click(_new)
    q_input.on_submit(_send)

    return _w.VBox(
        [header, chat, status, footer],
        layout=_w.Layout(
            border='1px solid #dee2e6', border_radius='8px',
            width='100%', box_shadow='0 2px 10px rgba(0,0,0,0.07)',
        ),
    )

display(make_geoai_chat())

## Bringing it all together — synthesis

You have now run the full GeoAI pipeline: spatial mapping, ecological correlation, individual-level logistic regression, DuckDB queries, and AI-assisted exploration. Exercise 5 asks you to synthesise these strands into a coherent epidemiological summary — the kind of "key messages" section you would write for a policy brief or a research paper.

Think about what the *pattern of results* tells you, not just individual numbers. Where are the findings consistent across levels of analysis? Where do they diverge? What would the next study need to do differently to move from association to evidence of effect? This reflection is the most transferable skill in the entire workshop.


---
### 🧪 Exercise 5 — Synthesis *(4 points)*

Write a 4–6 sentence epidemiological summary of what this analysis found. Address:
1. The spatial distribution of LBW across Zimbabwe and which provinces are highest
2. Which environmental factors show the clearest association with LBW and at which level of analysis
3. Why environmental effects were weaker in the individual-level logistic regression
4. Two key study limitations and how they could be addressed in future work

Your facilitator will mark this on: correct interpretation of ecological vs individual-level results,
understanding of confounding/selection bias/measurement error, and quality of suggested improvements.


*(Write your 4–6 sentence synthesis here)*


---
## 🏁 Workshop recap — What we built today

| Step | What | Tools |
|------|------|-------|
| NB01 | Loaded and mapped real Zimbabwe DHS 2015 birth records | pandas, matplotlib |
| NB02 | Explored CHIRPS/MODIS satellite climate data; saw El Niño 2015 signal | Python, matplotlib |
| NB03 | Explored Zimbabwe's climate landscape — seasonal cycles, spatial gradients, PM2.5 | pandas, matplotlib |
| NB04 | Joined health records to the environmental grid using cKDTree nearest-neighbour search | scipy.cKDTree |
| NB05 | Built trimester exposure windows; linked pregnancy timing to Zimbabwe's seasonal climate | pandas |
| NB06 | Environmental epidemiology: ecological + individual-level analysis, ORs, GeoAI | statsmodels, DuckDB |

**The central message:** Satellite-derived environmental data can be linked to population health
records at scale. The associations found here — greener provinces have lower LBW; hotter
clusters have higher LBW — are consistent with biological pathways (heat stress, nutritional
stress via vegetation). But study design and data quality determine what we can detect.
Environmental epidemiology requires careful attention to confounding, selection bias, and
exposure measurement error. These are research design problems that cannot be solved by
better satellite data alone.

---
*Data: Zimbabwe Demographic and Health Survey 2015 (DHS Program). Climate: simulated
exposure grids calibrated to Zimbabwe seasonal patterns and El Niño 2015 conditions.*


In [ ]:
%%html
<div id="nb06-scoreboard" style="font-family:system-ui,sans-serif;border:2px solid #adb5bd;border-radius:10px;padding:20px 24px;margin:24px 0;background:#f8f9fa">
  <div style="font-size:17px;font-weight:700;margin-bottom:12px;color:#1a1a2e">&#128202; Module 6 &#8212; Results &amp; GeoAI &#8212; Quiz Results</div>
  <div id="nb06-score-display"><p style="color:#856404;background:#fff3cd;padding:10px 12px;border-radius:6px;margin:0">&#9203; Complete all reflection questions above to see your score.</p></div>
  <div id="nb06-next-action" style="margin-top:14px"></div>
</div>
<script>
(function(){
  var QUESTIONS=[{id:'nb06q1',correct:'B'},{id:'nb06q2',correct:'B'},{id:'nb06q3',correct:'B'},{id:'nb06q4',correct:'C'}];
  var PASS=75;
  var NEXT=null;
  var NEXT_LABEL=null;
  var NBID='nb06';
  function update(){
    var correct=0,answered=0;
    QUESTIONS.forEach(function(q){
      var ans=localStorage.getItem('mcq_'+q.id);
      if(ans!==null){answered++;if(ans===q.correct)correct++;}
    });
    var total=QUESTIONS.length;
    var pct=total?Math.round(correct/total*100):0;
    var passed=pct>=PASS;
    var disp=document.getElementById(NBID+'-score-display');
    var act=document.getElementById(NBID+'-next-action');
    if(!disp)return;
    if(answered<total){
      disp.innerHTML='<p style="color:#856404;background:#fff3cd;padding:10px 12px;border-radius:6px;margin:0">&#9203; Answer all '+total+' questions above to see your score. ('+(total-answered)+' remaining)</p>';
      if(act)act.innerHTML='';
    }else if(passed){
      disp.innerHTML='<div style="background:#d4edda;color:#155724;padding:12px 16px;border-radius:8px"><b>&#10003; Score: '+correct+'/'+total+' ('+pct+'%) &#8212; MODULE PASSED</b><br><small>Your progress is saved &#8212; you can close and return to this notebook.</small></div>';
      if(act){
        localStorage.setItem(NBID+'_passed','true');
        if(NEXT){act.innerHTML='<a href="'+NEXT+'" style="display:inline-block;background:#198754;color:#fff;padding:10px 22px;border-radius:6px;text-decoration:none;font-weight:600;font-size:14px;margin-top:8px">&#10145; Continue to '+NEXT_LABEL+'</a>';}
        else{act.innerHTML='<div style="background:#cfe2ff;color:#084298;padding:12px 16px;border-radius:8px;margin-top:8px"><b>&#127881; You have completed the full GeoAI pipeline course! Congratulations.</b></div>';}
      }
    }else{
      disp.innerHTML='<div style="background:#f8d7da;color:#721c24;padding:12px 16px;border-radius:8px"><b>&#10007; Score: '+correct+'/'+total+' ('+pct+'%) &#8212; Below pass mark ('+PASS+'%)</b><br><small>Go back to any question and change your answer, then your score updates automatically.</small></div>';
      if(act)act.innerHTML='';
    }
  }
  document.addEventListener('mcq-update',update);
  if(document.readyState==='loading')document.addEventListener('DOMContentLoaded',update);
  else update();
  setInterval(update,3000);
})();
</script>

---

## 🎯 You have completed the GeoAI for Climate, Environment and Health pipeline

Starting from a raw DHS birth records file, you have:

1. **Cleaned and mapped** geocoded health data (NB01)
2. **Understood** the satellite provenance of your exposure data (NB02)
3. **Explored** satellite-derived climate products across Zimbabwe's landscape (NB03)
4. **Spatially joined** birth records to climate grid points (NB04)
5. **Constructed** trimester-specific exposure windows (NB05)
6. **Mapped, modelled, and queried** climate–health associations with GeoAI (NB06)

The methods you practised here are transferable to any setting where satellite data meets health records. Remote sensing professionals now have a route into health research. Health researchers now have a route into Earth observation.

That is the bridge this course was built to create.